Generate data trajectories $\boldsymbol{X}_{N+1}, \boldsymbol{U}_{N}$ on which the model rollout performance is evaluated

For each system generate trajectories based on...
- basic excitation signals (ramp, multisine, chirp, random-walk)
- further evaluation excitation patterns

These patterns are to be stored in `admpe.data_management.TESTING_TRAJECTORIES_ROOT`


**TODO:** Should all patterns be with or without noise? **Currently w/o noise**

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"

import pathlib
from functools import partial

import time
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import rc
rc('font',**{'family':'serif','serif':['Helvetica']})
mpl.rcParams['text.usetex'] = True
mpl.rcParams.update({'font.size': 10.95})
mpl.rcParams['text.latex.preamble']= (
    r"\usepackage{bm}\usepackage{amsmath}\usepackage{upgreek}\usepackage{times}"
)

figurewidth_pt = 433.62
figurewidth_in = figurewidth_pt / 72.27

print(figurewidth_in)

In [ ]:
import jax
jax.config.update("jax_platform_name", "cpu")
jax.config.update("jax_enable_x64", False)
import jax.numpy as jnp
import equinox as eqx 

In [ ]:
from admpe.data_management import TESTING_TRAJECTORIES_ROOT

from admpe.excitation.excitation_experiment_setup import setup_environment

---

In [ ]:
# setup envs:
env, penalty_function, featurize, env_params, model_class = setup_environment(system_name="cart_pole")
env = eqx.tree_at(lambda s: s.process_noise_variance, env, 0.0)

In [ ]:
env

### generate trajectories with simple excitation signals:

Should these be optimized to respect the constraints? Huh
Let us roll and see what happens


(ramp, multisine, chirp, random-walk, aprbs)

In [ ]:
from admpe.utils.signals import aprbs, multisine
from dmpe.models.model_utils import simulate_ahead_with_env
from dmpe.evaluation.plotting_utils import plot_feature_combinations

- start from a random starting point, then it actually makes some sense!!!

In [ ]:
seed = 12
batch_size = 100
n_steps  = 500

key = jax.random.PRNGKey(seed)

key, gen_key, env_key = jax.random.split(key, 3)

env_keys = jax.random.split(env_key, batch_size)
init_states = jax.vmap(
    lambda rng: env.init_state(rng=rng, deterministic_state=False)
)(env_keys)
init_obs = jax.vmap(env.generate_observation)(init_states)

actions = aprbs(n_steps=500, t_min=5, t_max=20, batch_size=100, key=gen_key)

plt.plot(actions[0])
plt.show()
plt.plot(actions[1])
plt.show()


simulate_batch = eqx.filter_jit(
    eqx.filter_vmap(
        simulate_ahead_with_env,
        in_axes=(None, 0, 0, 0),
    )
)

observations, last_states = simulate_batch(
    env, init_obs, init_states, actions
)

plot_feature_combinations(
    jnp.concatenate([observations[0, :-1], actions[0]], axis=-1),
    labels=np.arange(0, env.obs_dim + env.action_dim, 1),
)
plt.show()

In [ ]:
seed = 12
batch_size = 100
n_steps  = 500

key = jax.random.PRNGKey(seed)

key, gen_key, env_key = jax.random.split(key, 3)

env_keys = jax.random.split(env_key, batch_size)
init_states = jax.vmap(
    lambda rng: env.init_state(rng=rng, deterministic_state=False)
)(env_keys)
init_obs = jax.vmap(env.generate_observation)(init_states)

for transition_type in ["linear", "cosine", "quintic"]:
    actions = aprbs(
        n_steps=500,
        t_min=10,
        t_max=20,
        batch_size=100,
        key=gen_key,
        transition_type=transition_type,
        transition_steps=8
    )
    
    plt.plot(actions[0])
    plt.show()
    plt.plot(actions[1])
    plt.show()
    
    
    simulate_batch = eqx.filter_jit(
        eqx.filter_vmap(
            simulate_ahead_with_env,
            in_axes=(None, 0, 0, 0),
        )
    )
    
    observations, last_states = simulate_batch(
        env, init_obs, init_states, actions
    )
    
    # for i in range(10):
    #     plt.plot(observations[i])
    #     plt.show()
    plot_feature_combinations(
        jnp.concatenate([observations[0, :-1], actions[0]], axis=-1),
        labels=np.arange(0, env.obs_dim + env.action_dim, 1),
    )
    plt.show()

In [ ]:
seed = 12
batch_size = 100
n_steps  = 500

key = jax.random.PRNGKey(seed)

key, gen_key, env_key = jax.random.split(key, 3)

env_keys = jax.random.split(env_key, batch_size)
init_states = jax.vmap(
    lambda rng: env.init_state(rng=rng, deterministic_state=False)
)(env_keys)
init_obs = jax.vmap(env.generate_observation)(init_states)

actions, non_reduced = multisine(
    n_steps=500,
    fundamental_frequency=0.005,
    n_sines=5,
    batch_size=100,
    n_actions=1,
    key=gen_key,
    debug=True,
)

plt.plot(actions[0])
plt.show()



simulate_batch = eqx.filter_jit(
    eqx.filter_vmap(
        simulate_ahead_with_env,
        in_axes=(None, 0, 0, 0),
    )
)

observations, last_states = simulate_batch(
    env, init_obs, init_states, actions
)

# for i in range(10, 20):
#     plt.plot(observations[i])
#     plt.show()

plot_feature_combinations(
    jnp.concatenate([observations[0, :-1], actions[0]], axis=-1),
    labels=np.arange(0, env.obs_dim + env.action_dim, 1),
)
plt.show()

- continue for all other signals
- build a proper data base for these entries

- visualize more
- implement + test the other methods ~ Nelles2020 as source
- put the mathematical definitions into the appendix
- $\rightarrow$ put the data generation into a script
- iterate over seeds and data generation methods?
- build different signal lengths

In [ ]:
# load data from scripts -> checkout results. 
# incorporate into the model rollout tests

In [ ]:
from admpe.data_management import load_testing_sequences, list_modeling_results, load_modeling_result

In [ ]:
testing_data = load_testing_sequences("pendulum")

In [ ]:
from admpe.evaluation.modeling_evaluation import evaluate_model_rollout

In [ ]:
results = list_modeling_results(
    system_name=None,
    model_type="neural_euler",
    status="completed",
    exp_name="run1"
)

In [ ]:
modeling_result = load_modeling_result
wrapped_model = 

evaluate_model_rollout(
    wrapped_model,
    observations=testing_data["aprbs"]["observations"],
    actions=testing_data["aprbs"]["actions"],
)

### generate further excitation patterns that are only used for evaluation: